# Hybrid Search: Fusing pgvector + BM25 with RRF

**Hybrid retrieval** runs two searches over the same chunks and merges the results:

- **Dense** (pgvector) - semantic similarity via embeddings; catches paraphrases and meaning.
- **Sparse** (BM25 via `pg_textsearch`) - keyword ranking; nails exact terms, names, and rare words.

Each returns a *ranked list*. We combine them with **Reciprocal Rank Fusion (RRF)**, a simple, robust method that scores an item by its **rank** in each list (not by the raw, non-comparable scores). A chunk that ranks well on *either* signal rises to the top.

```
query --> dense (pgvector)  --> ranked list A --+
      \-> sparse (BM25)      --> ranked list B --+--> RRF fuse --> hybrid results
```

We reuse the `rag_documents` chunks ingested in `1_standard_rag/4_ocr_chunk_store_pgvector.ipynb`: pgvector has their embeddings, and notebook 4 also built a BM25 index (`bm25_chunks_idx`) over the same chunk text - we read both here.

Prerequisites: run notebook 4 to populate `rag_documents` and create `bm25_chunks_idx`, and have `pg_textsearch` installed (see `1_install_pgvector`).

## Dependencies

`langchain-postgres` + `langchain-openai` (dense side - calls the remote embedding model over its OpenAI-compatible API), `psycopg` (BM25 SQL), and `python-dotenv` - all already in `../requirements.txt`:

```bash
pip install -r requirements.txt
```

## Configuration and connections

Same `.env` as the other notebooks (in-cluster `PG_HOST=pgvector`). We open two things against the same database: a `psycopg` connection for the BM25 SQL, and a `PGVector` store for the dense side - using the **same embedding model and collection** as ingestion.

In [ ]:
import os
import psycopg
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

load_dotenv("../var.env")   # shared env file at day2skk/var.env

PG_HOST = os.environ.get("PG_HOST", "pgvector")
PG_PORT = os.environ.get("PG_PORT", "5432")
PG_USER = os.environ.get("PG_USER", "raguser")
PG_PASSWORD = os.environ.get("PG_PASSWORD", "change-me-please")
PG_DB = os.environ.get("PG_DB", "ragdb")
COLLECTION_NAME = "rag_documents"   # collection ingested in notebook 4
INDEX_NAME = "bm25_chunks_idx"      # BM25 index created in notebook 4

# Embedding model: the self-hosted Qwen3-Embedding-4B on vLLM. MUST match the
# model used at ingestion (1_standard_rag/4_ocr_chunk_store_pgvector.ipynb).
EMB_MODEL = os.environ.get("EMB_MODEL", "Qwen/Qwen3-Embedding-4B")
EMB_BASE_URL = os.environ.get("EMB_BASE_URL", "http://qwen3-emb-4b.default.svc.cluster.local/v1")
EMB_API_KEY = os.environ.get("EMB_API_KEY", "sk-dhU11z5FIjXQ8TLXpwDotpGetP21CQv3")

# psycopg connection for BM25 SQL
conn = psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"
)
conn.autocommit = True

# PGVector store for dense search (same embeddings + collection as notebook 4).
# check_embedding_ctx_length=False sends raw text to the model's own tokenizer.
embeddings = OpenAIEmbeddings(
    model=EMB_MODEL,
    base_url=EMB_BASE_URL,
    api_key=EMB_API_KEY,
    check_embedding_ctx_length=False,
)
connection_url = f"postgresql+psycopg://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
vector_store = PGVector(
    embeddings=embeddings,
    collection_name=COLLECTION_NAME,
    connection=connection_url,
    use_jsonb=True,
)
print("connected; collection:", COLLECTION_NAME)

## Locate the chunks and their BM25 index

`langchain-postgres` stores every chunk's text in the `document` column of its `langchain_pg_embedding` table, grouped by collection. Notebook 4 added the BM25 index `bm25_chunks_idx` over that column, so the *same* chunks are searchable both ways. Here we just confirm the extension and index exist, and look up the collection's id (so the BM25 query only scores this collection's rows).

(These `langchain_pg_*` table names are internal to `langchain-postgres`; they are stable within the 0.0.x line used here.)

In [ ]:
with conn.cursor() as cur:
    # extension present?
    cur.execute("SELECT 1 FROM pg_extension WHERE extname = 'pg_textsearch';")
    if cur.fetchone() is None:
        raise RuntimeError("pg_textsearch is not installed. See 1_install_pgvector.")

    # the BM25 index from notebook 4 exists?
    cur.execute("SELECT 1 FROM pg_indexes WHERE indexname = %s;", (INDEX_NAME,))
    if cur.fetchone() is None:
        raise RuntimeError(
            f"Index {INDEX_NAME!r} not found. "
            "Run 1_standard_rag/4_ocr_chunk_store_pgvector.ipynb first."
        )

    # id of our collection
    cur.execute("SELECT uuid FROM langchain_pg_collection WHERE name = %s;", (COLLECTION_NAME,))
    row = cur.fetchone()
    if row is None:
        raise RuntimeError(f"Collection {COLLECTION_NAME!r} not found. Run notebook 4 first.")
    COLLECTION_ID = row[0]

print("collection id:", COLLECTION_ID)
print(f"index {INDEX_NAME!r}: found")

## The two retrievers

**Dense** uses `PGVector.similarity_search`. **Sparse** uses the BM25 `<@>` operator, filtered to this collection (remember: BM25 scores are negative, so `ORDER BY ... <@> query` ascending gives the best first). Each returns a ranked list of chunk texts.

In [ ]:
def dense_search(query: str, k: int = 5) -> list[str]:
    return [d.page_content for d in vector_store.similarity_search(query, k=k)]


def sparse_search(query: str, k: int = 5) -> list[str]:
    sql = (
        "SELECT document "
        "FROM langchain_pg_embedding "
        "WHERE collection_id = %(cid)s "
        "ORDER BY document <@> to_bm25query(%(q)s, %(idx)s) "   # reads bm25_chunks_idx; ascending = best first
        "LIMIT %(k)s;"
    )
    with conn.cursor() as cur:
        cur.execute(sql, {"cid": COLLECTION_ID, "q": query, "idx": INDEX_NAME, "k": k})
        return [r[0] for r in cur.fetchall()]

## Reciprocal Rank Fusion (RRF)

RRF ignores the raw scores (which are not comparable between vector distance and BM25) and uses only the **rank** of each item in each list. An item at rank *r* (0-based) in a list contributes `1 / (rrf_k + r + 1)`; contributions are summed across lists. The constant `rrf_k` (commonly 60) softens the influence of top ranks so lower-ranked-but-agreed-upon items can still win.

Items are keyed by their chunk text, which is identical across both retrievers.

In [ ]:
def rrf_fuse(ranked_lists: list[list[str]], rrf_k: int = 60) -> list[tuple[str, float]]:
    scores: dict[str, float] = {}
    for ranked in ranked_lists:
        for rank, key in enumerate(ranked):
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank + 1)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)


def hybrid_search(query: str, k_each: int = 5, rrf_k: int = 60, top_n: int = 5):
    dense = dense_search(query, k_each)
    sparse = sparse_search(query, k_each)
    fused = rrf_fuse([dense, sparse], rrf_k=rrf_k)
    return fused[:top_n]

## Compare: dense vs sparse vs hybrid

Run all three for the same query. Dense and sparse often surface different chunks; the hybrid list blends them, keeping chunks that either method ranked highly.

In [ ]:
def preview(text: str, n: int = 80) -> str:
    return text[:n].replace("\n", " ")


query = "What is this document about?"

print(f"QUERY: {query!r}\n")

print("--- dense (pgvector) ---")
for i, t in enumerate(dense_search(query), 1):
    print(f"  {i}. {preview(t)}")

print("\n--- sparse (BM25) ---")
for i, t in enumerate(sparse_search(query), 1):
    print(f"  {i}. {preview(t)}")

print("\n--- hybrid (RRF) ---")
for i, (t, score) in enumerate(hybrid_search(query), 1):
    print(f"  {i}. (rrf={score:.4f}) {preview(t)}")

## Use it as a retriever

`hybrid_search` returns ranked chunk texts, which is exactly what a RAG chain feeds to the LLM as context. Dropping this in for the plain vector retriever in `5_standard_rag_langgraph.ipynb` upgrades that pipeline to **hybrid RAG** - the `retrieve` node calls `hybrid_search` instead of `similarity_search`, and everything downstream stays the same.

In [ ]:
def retrieve_context(query: str, k: int = 4) -> list[str]:
    """Top-k chunk texts by hybrid (dense + BM25) relevance - ready for a RAG prompt."""
    return [text for text, _ in hybrid_search(query, top_n=k)]


for chunk in retrieve_context("summarize the key points"):
    print("-", preview(chunk, 100))

## Recap

- **Hybrid retrieval** fuses **dense** (pgvector, semantic) and **sparse** (BM25 via `pg_textsearch`, keyword) search over the same chunks.
- Both read what notebook 4 produced: pgvector's embeddings and the **`bm25_chunks_idx`** index over the `document` column - so the two retrievers cover identical chunks (no re-indexing here). `to_bm25query('terms', 'bm25_chunks_idx')` names the index for the BM25 query.
- **RRF** merges the two ranked lists using only ranks (`1 / (rrf_k + rank)`), avoiding the problem that vector distances and BM25 scores are not directly comparable.
- Swap `hybrid_search` in for the vector retriever to turn the standard RAG graph into a **hybrid RAG** pipeline.

That completes hybrid retrieval: **pgvector for meaning + pg_textsearch/BM25 for keywords, fused with RRF.**